In [2]:
# Analise de artigo 32 do QR do recurso

import os, re
import json
import requests
import pandas as pd

In [3]:
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            try:
                data = response.json()
                json_data = json.dumps(data, indent=4)
                return(json_data)
            except Exception as e:
                return(f"Erro: {e}")
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")

In [3]:

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

numero = "PI0808715"
# atualiza no localhost as tres tabelas: carga, anterioridades e anterioridades_desc
comando = f"SELECT * FROM arquivados WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
print(resultado)

[(1011532, '1.1', 'PI0808715', datetime.date(2011, 8, 9), 'dialp', 0, 0), (1475660, '1.3', 'PI0808715', datetime.date(2014, 8, 12), 'dialp', 0, 0), (1501853, '6.6', 'PI0808715', datetime.date(2014, 9, 16), 'dialp', 0, 0), (2788968, '7.1', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 1), (2790019, '15.11', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 0), (2956330, '9.2', 'PI0808715', datetime.date(2017, 9, 19), 'dialp', 0, 2), (3015281, '12.2', 'PI0808715', datetime.date(2017, 12, 19), 'dialp', 0, 0)]


In [ ]:
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from carga where numero<>'NUMERO' and examinador='abrantes' and numero in (select numero from anterioridades_desc where artigo32='' and modelo='gpt-5-nano') and numero in (select numero from arquivados where despacho='12.2') "
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')

print(lista)

In [319]:
lista = ['numero','112015016028'] #     102013019765
print(len(lista))

2


In [ ]:
112014014731 nao pegou 214 e 200 Erro ao processar peticoes/112014014731_0000221406178378_260.txt
MU9100240 nao pegou 214 e 200 Erro ao processar peticoes/MU9100240_0000221100860600_200.txt
PI1103928 nao pegou 214 e 200 Erro ao processar peticoes/PI1103928_0000921107763370_200.txt
122017012058 não pegou 214 pegou 200 Erro ao processar peticoes/122017012058_0000221703598339_200.txt
112016018024 pegou 214 cortou reivindicação 3. Dispositivo, de acordo com qualquer uma das, pegou 200 cortou na 3, pegou na 260 UPDATE realizado
112015029135 pegou 214 200 260 UPDATE realizado
112015028917 pegou 214 e 200 260 UPDATE realizado
112015028880 pegou 214 200 200 UPDATE REALIZADO
122020017521 pegou 214 Erro ao processar peticoes/122020017521_29409161922956740_200.txt Erro ao processar peticoes/122020017521_29409161922956740_200
112015005153 pegou 214 200 Erro ao processar peticoes/112015005153_0000221500344626_200.txt: UPDATE REALIZADO
112015000416 pegou 214 reiv 10 cortou, nao pegou 200 mas pegou 260 UPDATE REALIZADO
102015011582 nao pegou 214 e 200 Erro ao processar peticoes/102015011582_0000221503550227_200.txt:
102013014262 pegou 214 e 200 mas nao pegou 260 UPDATE realizado
102012030377 nao pegou 214 e 200 Erro ao processar peticoes/102012030377_0000221208213649_200.txt
122020017517 nao pegou 214, 200, 260 Erro ao processar peticoes/122020017517_29409161922956316_200.txt:
112015002586 pegou 214 200 260 UPDATE REALIZADO
102015031507 pegou 214, 200 e nao pegou 260 UPDATE REALIZADO
102013019765 nao pegou 214 200 260 Erro ao processar peticoes/102013019765_0000221305520658_200.txt:
112015016028 nao pegou 214, pegou 200, 260

In [320]:
# atualiza artigo32 na tabela anterioridades_desc para modelo= gpt-5-nano
import re
import os, json
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from datetime import date
from langchain_openai import ChatOpenAI

hoje = date.today()

load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=openai_api_key,temperature=1)
hoje = date.today()

def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()
    
def limpar_linhas_irrelevantes(texto):
    linhas = texto.splitlines()
    linhas_filtradas = []
    for linha in linhas:
        linha_strip = linha.strip()
        # ❌ Remove linhas tipo: "Petição 870210110016, de 26/11/2021, pág. 55/69"
        if re.match(r'^Peti[cç][aã]o\s+\d+.*p[aá]g\.\s*\d+/\d+', linha_strip, re.IGNORECASE):
            continue
        # ❌ Remove linhas que são só número (ex: "5")
        if re.match(r'^\d+$', linha_strip):
            continue
        # ❌ Remove linhas tipo "5/15"
        if re.match(r'^\d+/\d+$', linha_strip):
            continue
        # ❌ Remove linhas vazias (opcional)
        if linha_strip == "":
            continue
        linhas_filtradas.append(linha)
    return "\n".join(linhas_filtradas)

def normalizar_texto(texto):
    # remove quebras no meio de palavras: "de ntre" → "dentre"
    texto = re.sub(r'(\w)\s+(\w)', r'\1\2', texto)
    # normaliza múltiplos espaços/quebras
    texto = re.sub(r'\s+', ' ', texto)
    return texto

def corrigir_numeracao_reivindicacao(texto: str) -> str:
    # corrige casos como 1o9), 2o), 3°), 4º)
    return re.sub(
        r'\b(\d+)(?:[oOº°]+|\D+\d+)\)', 
        r'\1)', 
        texto
    )

def extrair_reivindicacoes(texto):
    texto = limpar_linhas_irrelevantes(texto)
    texto = texto.replace('°','').replace('o)',')').replace('º','')
    #texto = corrigir_numeracao_reivindicacao(texto)
    # pega só o bloco de reivindicações
    bloco = re.search(
        r'^\s*R\s*E\s*I\s*V\s*I\s*N\s*D\s*I\s*C\s*A\s*\w*\w+\s*:?\s*\n+(.*?)(?=^\s*(?:METODO|M[ÉE]TODO|FIGURAS|FUNDAMENTOS|DESCRI[CÇ][AÃ]O|RESUMO|RELAT[ÓO]RIO)\b|\Z)',
        texto,
        re.DOTALL | re.MULTILINE
    )        
    resultado = []
    if bloco:
        texto_reiv  = bloco.group(1)
        #print(texto_reiv)
        inicio = re.search(r'^\s*1\s*[\.\)o]', texto_reiv, re.MULTILINE | re.IGNORECASE)
        if inicio:
            texto_reiv  = texto_reiv [inicio.start():]
        padrao = re.compile(
            r'^\s*(\d+)\s*[\.\)]\s*'
            r'(.*?)'
            r'(?=^\s*\d+\s*[\.\)]|'                          
            r'\n\s*(?:METODO|INSTRUCOES|FIGURAS|FUNDAMENTOS|DESCRIÇÃO|DESCRICAO|RESUMO|RELATORIO|RELATÓRIO)\b|'  
            r'\Z)',
            re.DOTALL | re.MULTILINE
        )    
        padrao = re.compile(
            r'(?<!\d)(\d+)\s*[\.\)]\s*'     # início (1), 2), etc.
            r'([\s\S]*?)'
            r'(?=\n?\s*(?:\d+\s*[\.\)])|\Z)',  # próxima reivindicação REAL
            re.DOTALL
        )
        padrao = re.compile(
            r'(?:^|\n)\s*(\d+)\s*[\.\)]\s*'
            r'([\s\S]*?)'
            r'(?=(?:\n\s*\d+\s*[\.\)])|\Z)',
            re.DOTALL
        )
        resultado = []
        for m in padrao.finditer(texto_reiv):
            numero = m.group(1)
            conteudo = m.group(2).strip()
            conteudo = (
                conteudo
                .replace('\r', ' ')
                .replace('\xa0', ' ')
                .replace('\ufeff', ' ')
                .replace('"',' ')
                .replace("'",' ')
                .replace('\n',' ')
                .replace('  ',' ')
            )
            resultado.append(f"{numero}. {conteudo}")
            
    if resultado is not None:
        return resultado 
    else:
        return None
        
        
    
#você quer navegar sequencialmente no texto, não só pegar datas soltas. A regra é:
#Começa em "Quadro 1"
#Pega a primeira data
#Vai até "Quadro Reivindicatório"
#Continua até a segunda data
#Se aparecer "Desenhos" antes da segunda data → retorna a primeira
#Senão → retorna a segunda
def detectar_data_indeferimento(texto: str):
    # normaliza (evita problemas com maiúsculas/minúsculas)
    texto_lower = texto.lower()
    # 1. encontrar início em "quadro 1"
    inicio_match = re.search(r'quadro\s*1', texto_lower)
    if not inicio_match:
        return None
    trecho = texto[inicio_match.start():]
    # 2. encontrar todas as datas no trecho
    datas = list(re.finditer(r'\d{2}/\d{2}/\d{4}', trecho))
    if len(datas) < 2:
        return None
    primeira_data = datas[0]
    segunda_data = datas[1]
    #print(f"{primeira_data} {segunda_data}")
    # 3. encontrar "quadro reivindicatório"
    qr_match = re.search(r'quadro\s*reivindicat[óo]rio', trecho, re.IGNORECASE)
    if not qr_match:
        return None
    #print(qr_match)
    pos_qr = qr_match.end()
    # 4. pegar trecho entre QR e segunda data
    trecho_entre = trecho[pos_qr:segunda_data.start()]
    # 5. verificar se existe "desenhos" nesse intervalo
    if re.search(r'desenhos', trecho_entre, re.IGNORECASE):
        return primeira_data.group()
    else:
        return segunda_data.group()

def converter_data(data_iso):
    ano, mes, dia = data_iso.split("-")
    return f"{dia}/{mes}/{ano}"
    
arquivo_entrada = 'resolucao93.txt'
resolucao_93 = ''
with open(arquivo_entrada, 'r', encoding='utf-8') as f:
    resolucao_93 = f.read()

data = {}
data["patents"] = lista
with open("descricao.sql", "a", encoding="utf-8") as f1:
    for i in range(1, len(data["patents"])):
        numero = data["patents"][i] 
        json_data = None
        
        reivindicacoes_214 = ''
        data_peticao_recurso = None
        tipo = '214'
        query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and tipo_peticao='{tipo}' order by data_peticao desc"+'"' 
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
        print(url)
        numnossonumero = None
        try:
            json_data = conectar_siscap(url,return_json=True)
        except:
            pass
        if json_data:
            try:
                json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                json_data = json_data.replace('\r', '')
                data_pedido = json.loads(json_data)
                patents = data_pedido.get("patents", [])
                if not patents:
                    continue
                primeiro = patents[0]  # for primeiro in patents:
                numnossonumero = primeiro.get("numnossonumero")
                data_peticao_recurso = primeiro.get("data_peticao")
                cd_imagem = primeiro.get("cd_imagem")
                arquivo_entrada = f"peticoes/{numero}_{numnossonumero}_{tipo}.txt"
                conteudo = ''
                if os.path.exists(arquivo_entrada):
                    with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                        conteudo = f.read()
                        reivindicacoes_214 = extrair_reivindicacoes(conteudo)
                    if reivindicacoes_214:
                        print(f"=== REIVINDICAÇÕES EXTRAÍDAS NO 214 {data_peticao_recurso} ===\n")
                        print(reivindicacoes_214)
                    else:
                        print(f"Termo 'REIVINDICACOES' não encontrado na petição 214 {data_peticao_recurso}.")
                else:
                    print(f"não encontrei {arquivo_entrada} {data_peticao_recurso} [{cd_imagem}]")
            except Exception as e:
                print(f"Erro ao processar {arquivo_entrada}: {e}")

        #break
        reivindicacoes_200 = ''
        data_200 = None
        tipo = '200'
        query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and tipo_peticao='{tipo}'"+'"' 
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
        print(url)
        numnossonumero = None
        try:
            json_data = conectar_siscap(url,return_json=True)
        except:
            pass
        if json_data:
            try:
                json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                json_data = json_data.replace('\r', '')
                data_pedido = json.loads(json_data)
                patents = data_pedido.get("patents", [])
                if patents:
                    for primeiro in patents:
                        numnossonumero = primeiro.get("numnossonumero")
                        cd_imagem = primeiro.get("cd_imagem")
                        data_200 = primeiro.get("data_peticao")
                        arquivo_entrada = f"peticoes/{numero}_{numnossonumero}_{tipo}.txt"
                        print(arquivo_entrada)
                        conteudo = ''
                        if os.path.exists(arquivo_entrada):
                            with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                                conteudo = f.read()
                                reivindicacoes_200 = extrair_reivindicacoes(conteudo)
                            if reivindicacoes_200:
                                print(f"=== REIVINDICAÇÕES EXTRAÍDAS NO 200 {data_200}===\n")
                                print(reivindicacoes_200)
                            else:
                                print(f"Termo 'REIVINDICACOES' não encontrado na petição 200 {data_200}.")
                        else:
                            print(f"não encontrei {arquivo_entrada} {data_200} [{cd_imagem}]")
    
            except Exception as e:
                print(f"Erro ao processar petição 200: {e}")

# SELECT * FROM despachos_pag d WHERE d.numero = '112021023943' AND TRIM(d.tipo_peticao) IN ('260') AND EXISTS ( SELECT 1 FROM despachos_pag WHERE numero = d.numero AND TRIM(tipo_peticao) = '200' ) AND EXISTS ( SELECT 1 FROM despachos_pag WHERE numero = d.numero AND TRIM(tipo_peticao) IN ('203','204','205') ) ORDER BY d.data_peticao;

        #break
        data_pedido_exame = None
        query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and tipo_peticao in ('203','204','205')"+'"' 
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
        print(url)
        numnossonumero = None
        try:
            json_data = conectar_siscap(url,return_json=True)
        except:
            pass
        if json_data:
            try:
                json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                json_data = json_data.replace('\r', '')
                data_pedido = json.loads(json_data)
                patents = data_pedido.get("patents", [])
                if not patents:
                    continue
                for primeiro in patents:
                    data_pedido_exame = primeiro.get("data_peticao")
                    tipo = primeiro.get("tipo_peticao")
                    print(f"Pedido de exame: {data_pedido_exame}")
            except Exception as e:
                print(f"Erro ao processar data de pedido de exame: {e}")


        reivindicacoes_emendas = None
        data_valida_pedido_exame = None
        if data_200:
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and (data_peticao>'{data_200}' and data_peticao<='{data_pedido_exame}') and tipo_peticao='260' order by data_peticao desc"+'"' 
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            numnossonumero = None
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                pass
            if json_data:
                try:
                    json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                    json_data = json_data.replace('\r', '')
                    data_pedido = json.loads(json_data)
                    patents = data_pedido.get("patents", [])
                    if not patents:
                        continue
                    for primeiro in patents: # pega o primeiro registro ou seja, a útlma petição válida antes do 203
                        data_emenda = primeiro.get("data_peticao")
                        tipo_emenda = primeiro.get("tipo_peticao")
                        cd_imagem = primeiro.get("cd_imagem")
                        numnossonumero = primeiro.get("numnossonumero")
                        print(f"Emendas: {data_emenda} tipo {tipo_emenda}")
                        arquivo_entrada = f"peticoes/{numero}_{numnossonumero}_{tipo_emenda}.txt"
                        print(arquivo_entrada)
                        conteudo = ''
                        if os.path.exists(arquivo_entrada):
                            with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                                conteudo = f.read()
                                reivindicacoes_emendas = extrair_reivindicacoes(conteudo)
                            if reivindicacoes_emendas:
                                print(f"=== REIVINDICAÇÕES EXTRAÍDAS NO 260 {data_emenda}===\n")
                                print(reivindicacoes_emendas)
                                break
                            else:
                                print(f"Termo 'REIVINDICACOES' de emendas não encontrado na petição 260. {data_emenda}")
                        else:
                            print(f"não encontrei {arquivo_entrada} {data_emenda} [{cd_imagem}]")
        
                except Exception as e:
                    print(f"Erro ao processar {numero} petição 260: {e}")

        #break
        reivindicacoes_validas = reivindicacoes_200
        data_valida_pedido_exame = data_200
        if reivindicacoes_emendas:
            reivindicacoes_validas = reivindicacoes_emendas
            data_valida_pedido_exame = data_emenda
            
        data_indeferimento = None
        data_QR_indeferido_lido = None
        query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where numero='{numero}' and decisao in ('indeferimento','9.2') and anulado=0"+'"' 
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
        print(url)
        numnossonumero = None
        try:
            json_data = conectar_siscap(url,return_json=True)
        except:
            pass
        if json_data:
            try:
                json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                json_data = json_data.replace('\r', '')
                data_pedido = json.loads(json_data)
                patents = data_pedido.get("patents", [])
                if not patents:
                    continue
                for primeiro in patents:
                    data_indeferimento = primeiro.get("rpi")
                    divisao = primeiro.get("divisao")
                    codigo = primeiro.get("codigo")
                    caminho_do_arquivo = f"pareceres/{numero}{codigo}.txt"
                    print(f"Indeferimento: {caminho_do_arquivo}")
                    if os.path.exists(caminho_do_arquivo):
                        with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                            texto_relatorio = arquivo.read()
                        data_QR_indeferido_lido =  detectar_data_indeferimento(texto_relatorio)
                        print(f"QR indeferido lido = {data_QR_indeferido_lido}")
                    else:
                        print(f"Arquivo indeferimento não encontrado: {caminho_do_arquivo}")
                        texto_relatorio = None  # ou "" dependendo do seu uso
                    
            except Exception as e:
                print(f"Erro ao processar indeferimento: {e}")

        #break
        reivindicacoes_indeferimento = None
        data_QR_indeferimento = None
        if data_pedido_exame:
            reivindicacoes_indeferimento = reivindicacoes_validas
            data_QR_indeferimento = data_valida_pedido_exame
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and (data_peticao>'{data_pedido_exame}' and data_peticao<='{data_indeferimento}') and tipo_peticao in ('260','207','281') order by data_peticao desc"+'"' 
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            numnossonumero = None
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                pass
            if json_data:
                try:
                    json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                    json_data = json_data.replace('\r', '')
                    data_pedido = json.loads(json_data)
                    patents = data_pedido.get("patents", [])
                    if not patents:
                        continue
                    for primeiro in patents:
                        data_QR_indeferimento_teste = primeiro.get("data_peticao")
                        tipo_QR_indeferimento = primeiro.get("tipo_peticao")
                        cd_imagem = primeiro.get("cd_imagem")
                        numnossonumero = primeiro.get("numnossonumero")
                        print(f"QR indeferimento testando: {data_QR_indeferimento_teste} tipo {tipo_QR_indeferimento}")
                        arquivo_entrada = f"peticoes/{numero}_{numnossonumero}_{tipo_QR_indeferimento}.txt"
                        print(arquivo_entrada)
                        conteudo = ''
                        if os.path.exists(arquivo_entrada):
                            with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                                conteudo = f.read()
                                reivindicacoes_indeferimento = extrair_reivindicacoes(conteudo)
                            if reivindicacoes_indeferimento:
                                print(f"=== REIVINDICAÇÕES EXTRAÍDAS DO QR INDEFERIMENTO {data_QR_indeferimento_teste} ===\n")
                                print(reivindicacoes_indeferimento)
                                data_QR_indeferimento = data_QR_indeferimento_teste
                                break # encontrei entao não precisa pesquisar mais
                            else:
                                print(f"Termo 'REIVINDICACOES' não encontrado do QR indeferimento {data_QR_indeferimento_teste}.")
                        else:
                            print(f"não encontrei {arquivo_entrada} {data_QR_indeferimento} [{cd_imagem}]")
                        
                except Exception as e:
                    print(f"Erro ao processar {numero} QR indeferimento: {e}")

        data_peticao_recurso = converter_data(data_peticao_recurso)
        data_valida_pedido_exame = converter_data(data_valida_pedido_exame)
        if reivindicacoes_214:
            introducao = f"Os quadros comparados são o quadro reivindicatório de {data_valida_pedido_exame} válido quando do pedido de exame e o QR apresentado no recurso {data_peticao_recurso}. "
        else:
            introdução = "Não teve apresentação de novo quadro reivindicatório na petição 214, logo não há comparação a ser feita"
        print(introducao)   
        #break
        if reivindicacoes_214 and reivindicacoes_validas:
            query = f"""Compare o quadro reivindicatório válido [{reivindicacoes_validas}] com o 
            quadro reivindicatório da fase recursal [{reivindicacoes_214}] e determine se esta emenda atende a Resolução 93 do INPI [{resolucao_93}]. 
            Confronte o QR Recursal com o QR Válido. APLIQUE A PREVALÊNCIA DA CARACTERIZANTE: Se a categoria técnica real estava na parte caracterizante, 
            o ajuste do preâmbulo é permitido. ACUSE qualquer termo novo que não seja uma restrição clara ou correção de erro material. Por exemplo 
            considere que QR válido pleiteava uma cadeira genérica, se a emenda recursal apresenta QR pedindo cadeira de couro trata-se de uma emenda 
            restritiva, que é permitida. Se fosse invertido essa ampliação seria indevida. Comece inicialmente comparando a reivindicação 1 do QR válido 
            com a reivindicação 1 do recurso e depois prossiga comparando com as demais reivindicações independentes. Depois leve em conta também 
            as reivindicações dependentes. Apresente como resultado a comparação dos QRs sem fazer qualquer introdução sobre a Resolução 93. 
            Não apresente qualquer sugestão de correção do quadro reivindicação, limite-se a apresentar suas conclusões quanto ao enquadramento
            no artigo 32 da LPI. Nada de finalizar com recomendações do tipo: Se quiser, eu posso ... Se concentre na justificativa para sua conclusão 
            se fere artigo 32 da LPI ou não. Não reproduza textualmente a reivindicação, apenas trechos, se necessário. Limite a resposta em no máximo 10 linhas.
            """
            try:
                messages=[{"role":"user", "content": query}]
                response = llm.invoke(messages)
                resposta = response.content
                texto_corrigido = " ".join(resposta.split())
                texto_corrigido = texto_corrigido.replace("'", "")
                texto_corrigido = texto_corrigido.replace('"', "")
                texto_corrigido = introducao + texto_corrigido
                sql_resumo = f"UPDATE anterioridades_desc SET artigo32='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
                print(sql_resumo)
                f1.write(sql_resumo + "\n")
            except Exception as e:
                print("ERRO NO PRIMEIRO LLM:", e)

        print(f"continuando agora testando QR exame e QR indeferido... data valida pedido exame {data_valida_pedido_exame}")
        if reivindicacoes_indeferimento is None:
            reivindicacoes_indeferimento = reivindicacoes_validas
            data_QR_indeferimento = data_valida_pedido_exame
            
        erro_quadro_I = ''
        data_QR_indeferimento = converter_data(data_QR_indeferimento)
        if data_QR_indeferimento!=data_QR_indeferido_lido:
            erro_quadro_I = f"O Quadro I aponta a data da petição com as reivindicações de {data_QR_indeferido_lido}, mas a petição correta é a de {data_QR_indeferimento}."
        introducao = f"Os quadros comparados são o quadro reivindicatório de {data_valida_pedido_exame} válido quando do pedido de exame e o QR válido no indeferimento {data_QR_indeferimento}. " 
        introducao = introducao + erro_quadro_I
        if data_valida_pedido_exame==data_QR_indeferimento:
            introducao = introducao + "Como o QR do indeferimento é o mesmo do pedido de exame não há comparação a ser feita. "
            sql_resumo = f"UPDATE anterioridades_desc SET artigo32_indeferimento='{introducao}', data='{hoje}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f1.write(sql_resumo + "\n")
        print(introducao)
        if reivindicacoes_indeferimento and reivindicacoes_validas and (data_valida_pedido_exame!=data_QR_indeferimento):
            query = f"""Agora ignore o quadro reivindicatório do recurso. Compare o quadro reivindicatório válido quando do pedido de exame [{reivindicacoes_validas}] com o 
            quadro reivindicatório analisado no indeferimento [{reivindicacoes_indeferimento}] e determine se esta emenda atende a Resolução 93 do INPI [{resolucao_93}]. 
            Confronte o QR de indeferimento com o QR Válido quando do pedido de exame. APLIQUE A PREVALÊNCIA DA CARACTERIZANTE: Se a categoria técnica real estava na parte caracterizante, 
            o ajuste do preâmbulo é permitido. ACUSE qualquer termo novo que não seja uma restrição clara ou correção de erro material. Por exemplo 
            considere que QR válido pleiteava uma cadeira genérica, se o indeferimento apresenta QR pedindo cadeira de couro trata-se de uma emenda 
            restritiva, que é permitida. Se fosse invertido essa ampliação seria indevida. Comece inicialmente comparando a reivindicação 1 do QR válido 
            com a reivindicação 1 do indeferimento e depois prossiga comparando com as demais reivindicações independentes. Depois leve em conta também 
            as reivindicações dependentes. Apresente como resultado a comparação dos QRs sem fazer qualquer introdução sobre a Resolução 93. 
            Não apresente qualquer sugestão de correção do quadro reivindicatório, limite-se a apresentar suas conclusões quanto ao enquadramento
            no artigo 32 da LPI. Nada de finalizar com recomendações do tipo: Se quiser, eu posso ... Se concentre na justificativa para sua conclusão 
            se fere artigo 32 da LPI ou não. Não reproduza textualmente a reivindicação, apenas trechos, se necessário. Limite a resposta em no máximo 10 linhas."""

            try:
                messages=[{"role":"user", "content": query}]
                response = llm.invoke(messages)
                resposta = response.content
                texto_corrigido = " ".join(resposta.split())
                texto_corrigido = texto_corrigido.replace("'", "")
                texto_corrigido = texto_corrigido.replace('"', "")
                texto_corrigido = introducao + texto_corrigido
                sql_resumo = f"UPDATE anterioridades_desc SET artigo32_indeferimento='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
                print(sql_resumo)
                f1.write(sql_resumo + "\n")
            except Exception as e:
                print("ERRO NO SEGUNDO LLM:", e)

        print('#################')
        if len(reivindicacoes_validas) > 0:
            reivindicacoes_validas = [t.replace('"', ' ') for t in reivindicacoes_validas]
            sql_resumo = f"UPDATE anterioridades_desc SET reiv_exame=" + '"' + f"{reivindicacoes_validas}" + '"' + f" WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f1.write(sql_resumo + "\n")
        
        if len(reivindicacoes_indeferimento) > 0:
            reivindicacoes_indeferimento = [t.replace('"', ' ') for t in reivindicacoes_indeferimento]
            sql_resumo = f"UPDATE anterioridades_desc SET reiv_indeferimento=" + '"' + f"{reivindicacoes_indeferimento}" + '"' + f" WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f1.write(sql_resumo + "\n")

        if len(reivindicacoes_214) > 0:
            reivindicacoes_214 = [t.replace('"', ' ') for t in reivindicacoes_214]
            sql_resumo = f"UPDATE anterioridades_desc SET reiv_recurso=" + '"' + f"{reivindicacoes_214}" + '"' + f" WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f1.write(sql_resumo + "\n")
        else:
            sql_resumo = f"UPDATE anterioridades_desc SET reiv_recurso='O recurso 214 não apresentou quadro reivindicatório' WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f1.write(sql_resumo + "\n")


https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag where numero='112015016028' and tipo_peticao='214' order by data_peticao desc"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Termo 'REIVINDICACOES' não encontrado na petição 214 2022-03-03.
https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag where numero='112015016028' and tipo_peticao='200'"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


peticoes/112015016028_0000921504998129_200.txt
=== REIVINDICAÇÕES EXTRAÍDAS NO 200 2015-07-02===

['1. Metodo implementado por computador para mitigar per- das por fraude durante uma transacao iniciada com c artao de paga- mento, o metodo implementado utilizando um disposit ivo de computa- dor acoplado a um dispositivo de memoria, o metodo caracterizado por compreender: receber uma mensagem de solicitacao de autorizacao que solicita a autorizacao de uma transacao, a transacao iniciada utilizando um cartao de pagamento que inclui um primeiro dispo sitivo de segu- ranca operavel para transacoes iniciadas dentro de uma regiao geo- grafica predefinida e um segundo dispositivo de seg uranca operavel para transacoes iniciadas tanto dentro da regiao ge ografica predefinida quanto fora da regiao geografica predefinida, em qu e a transacao e iniciada fora da regiao geografica predefinida, a m ensagem de solicita- cao de autorizacao incluindo primeiros dados de car tao de pagamento adquiridos do

...

Emendas: 2015-08-05 tipo 260
peticoes/112015016028_0000921505490498_260.txt
=== REIVINDICAÇÕES EXTRAÍDAS NO 260 2015-08-05===

['1. Metodo implementado por computador para mitigar per- das por fraude durante uma transacao iniciada com c artao de paga- mento, o metodo implementado utilizando um disposit ivo de computa- dor acoplado a um dispositivo de memoria, o metodo caracterizado por compreender: receber uma mensagem de solicitacao de autorizacao que solicita a autorizacao de uma transacao, a transacao iniciada utilizando um cartao de pagamento que inclui um primeiro dispo sitivo de segu- ranca operavel para transacoes iniciadas dentro de uma regiao geo- grafica predefinida e um segundo dispositivo de seg uranca operavel para transacoes iniciadas tanto dentro da regiao ge ografica predefinida quanto fora da regiao geografica predefinida, em qu e a transacao e iniciada fora da regiao geografica predefinida, a m ensagem de solicita- cao de autorizacao incluindo primeiros dados de car t

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Indeferimento: pareceres/1120150160281521512.txt
QR indeferido lido = None
https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag where numero='112015016028' and (data_peticao>'2016-06-29' and data_peticao<='2022-01-04') and tipo_peticao in ('260','207','281') order by data_peticao desc"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


QR indeferimento testando: 2021-12-29 tipo 207
peticoes/112015016028_29409161944449123_207.txt
Termo 'REIVINDICACOES' não encontrado do QR indeferimento 2021-12-29.
QR indeferimento testando: 2020-04-28 tipo 207
peticoes/112015016028_29409161917913493_207.txt
=== REIVINDICAÇÕES EXTRAÍDAS DO QR INDEFERIMENTO 2020-04-28 ===

['1. Metodo implementado por computador para mitigar pe r- das por fraude durante uma transacao iniciada em um ATM (118) que esta localizado fora de uma regiao geopolitica predefinida associada a um emissor (30) de um cartao de pagamento usado na transacao, o emissor (30) localizado dentro de e mantendo uma rede de processa- mento na regiao geopolitica predefinida , o metodo implementado utili- zando um dispositivo de comput ador de autorizacao acoplado ao ATM por uma rede d e intercambio separada da rede de processamento do emissor , o metodo caracterizado p elo fato de que compreende:  receber , a partir do ATM pela rede de intercambio, uma mensagem de solicitacao 

In [ ]:

# detecta as petições com cd_iamgem = 0
# SELECT d.* FROM despachos_pag d JOIN arquivados a ON a.numero = d.numero AND a.despacho = '12.2' JOIN carga c ON c.numero = d.numero WHERE d.cd_imagem IN (0,1);
# 2263 petições com cd_imagem = 0 ou 1
# SELECT d.* FROM despachos_pag d JOIN arquivados a ON a.numero = d.numero AND a.despacho = '12.2' JOIN carga c ON c.numero = d.numero AND c.examinador='abrantes' AND tipo_peticao IN ('200','207','210','214','260','272','280','281') WHERE d.cd_imagem IN (0,1);
# 27 petições

query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga " + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
print(url)
numnossonumero = None
try:
    json_data = conectar_siscap(url,return_json=True)
except:
    pass
if json_data:
    try:
        json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
        json_data = json_data.replace('\r', '')
        data_pedido = json.loads(json_data)
        patents = data_pedido.get("patents", [])
        if patents:
            for primeiro in patents:
                numero = primeiro.get("numero")
                query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and (cd_imagem=0 or cd_iamgem=1)"+'"' 
                url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
                numnossonumero = None
                try:
                    json_data = conectar_siscap(url,return_json=True)
                except:
                    pass
                if json_data:
                    try:
                        json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
                        json_data = json_data.replace('\r', '')
                        data_pedido = json.loads(json_data)
                        patents = data_pedido.get("patents", [])
                        if not patents:
                            continue
                        for primeiro in patents:
                            numnossonumero = primeiro.get("numnossonumero")
                            data_peticao = primeiro.get("data_peticao")
                            cd_imagem = primeiro.get("cd_imagem")
                            tipo = primeiro.get("tipo_peticao")
                            arquivo_entrada = f"peticoes/{numero}_{numnossonumero}_{tipo}.txt"
                            print(f"{arquivo_entrada} {data_peticao}")
                    except Exception as e:
                        print(f"Erro (1) ao processar {numero}: {e}")
    except Exception as e:
        print(f"Erro (2) ao processar {numero}: {e}")

In [25]:
texto = """
Quadro 1.

Quadro 1  Páginas do pedido examinadas
                Elemento                               Páginas               n.o da Petição           Data

Relatório Descritivo                               27/68 a 62/68
                                                                             870230021203           13/03/2023
Quadro Reivindicatório                             63/68 a 68/68
Desenhos                                                 67/89
                                                                             870210071194           05/08/2021
Resumo                                             68/89 a 86/89
"""

def detectar_data_indeferimento(texto: str):
    # normaliza (evita problemas com maiúsculas/minúsculas)
    texto_lower = texto.lower()
    # 1. encontrar início em "quadro 1"
    inicio_match = re.search(r'quadro\s*1', texto_lower)
    if not inicio_match:
        return None
    trecho = texto[inicio_match.start():]
    # 2. encontrar todas as datas no trecho
    datas = list(re.finditer(r'\d{2}/\d{2}/\d{4}', trecho))
    if len(datas) < 2:
        return None
    primeira_data = datas[0]
    segunda_data = datas[1]
    #print(f"{primeira_data} {segunda_data}")
    # 3. encontrar "quadro reivindicatório"
    qr_match = re.search(r'quadro\s*reivindicat[óo]rio', trecho, re.IGNORECASE)
    if not qr_match:
        return None
    #print(qr_match)
    pos_qr = qr_match.end()
    # 4. pegar trecho entre QR e segunda data
    trecho_entre = trecho[pos_qr:segunda_data.start()]
    # 5. verificar se existe "desenhos" nesse intervalo
    if re.search(r'desenhos', trecho_entre, re.IGNORECASE):
        return primeira_data.group()
    else:
        return segunda_data.group()

print(detectar_data_indeferimento(texto))


13/03/2023


In [314]:
def limpar_linhas_irrelevantes(texto):
    linhas = texto.splitlines()
    linhas_filtradas = []
    for linha in linhas:
        linha_strip = linha.strip()
        # ❌ Remove linhas tipo: "Petição 870210110016, de 26/11/2021, pág. 55/69"
        if re.match(r'^Peti[cç][aã]o\s+\d+.*p[aá]g\.\s*\d+/\d+', linha_strip, re.IGNORECASE):
            continue
        # ❌ Remove linhas que são só número (ex: "5")
        if re.match(r'^\d+$', linha_strip):
            continue
        # ❌ Remove linhas tipo "5/15"
        if re.match(r'^\d+/\d+$', linha_strip):
            continue
        # ❌ Remove linhas vazias (opcional)
        if linha_strip == "":
            continue
        linhas_filtradas.append(linha)
    return "\n".join(linhas_filtradas)

def normalizar_texto(texto):
    # remove quebras no meio de palavras: "de ntre" → "dentre"
    texto = re.sub(r'(\w)\s+(\w)', r'\1\2', texto)
    # normaliza múltiplos espaços/quebras
    texto = re.sub(r'\s+', ' ', texto)
    return texto

def corrigir_numeracao_reivindicacao(texto: str) -> str:
    # corrige casos como 1o9), 2o), 3°), 4º)
    return re.sub(
        r'\b(\d+)(?:[oOº°]+|\D+\d+)\)', 
        r'\1)', 
        texto
    )
    
def extrair_reivindicacoes(texto):
    texto = limpar_linhas_irrelevantes(texto)
    texto = texto.replace('°','').replace('o)',')').replace('º','')
    #texto = corrigir_numeracao_reivindicacao(texto)
    # pega só o bloco de reivindicações
    bloco = re.search(
        r'REIVINDICA[CÇ][OÕ]ES\s*:?\s*(.*?)(?=\n\s*(?:METODO|FUNDAMENTOS|DESCRIÇÃO|DESCRICAO|RESUMO|RELATORIO|RELATÓRIO)\b|\Z)',
        texto,
        re.DOTALL
    )
    bloco = re.search(
        r'^\s*REIVINDICA[CÇ][OÕ]ES\s*:?\s*\n+(.*?)(?=^\s*(?:METODO|M[ÉE]TODO|FUNDAMENTOS|DESCRI[CÇ][AÃ]O|RESUMO|RELAT[ÓO]RIO)\b|\Z)',
        texto,
        re.DOTALL | re.MULTILINE
    )        
    bloco = re.search(
        r'^\s*R\s*E\s*I\s*V\s*I\s*N\s*D\s*I\s*C\s*A\s*\w*\w+\s*:?\s*\n+(.*?)(?=^\s*(?:METODO|M[ÉE]TODO|FIGURA|FUNDAMENTOS|DESCRI[CÇ][AÃ]O|RESUMO|RELAT[ÓO]RIO)\b|\Z)',
        texto,
        re.DOTALL | re.MULTILINE
    )        
    resultado = []
    if bloco:
        
        texto_reiv  = bloco.group(1)
        #print(texto_reiv)
        inicio = re.search(r'^\s*1\s*[\.\)o]', texto_reiv, re.MULTILINE | re.IGNORECASE)
        if inicio:
            texto_reiv  = texto_reiv [inicio.start():]
        padrao = re.compile(
            r'^\s*(\d+)\s*[\.\)]\s*'
            r'(.*?)'
            r'(?=^\s*\d+\s*[\.\)]|'                          
            r'\n\s*(?:METODO|FUNDAMENTOS|DESCRIÇÃO|DESCRICAO|RESUMO|RELATORIO|RELATÓRIO)\b|'  
            r'\Z)',
            re.DOTALL | re.MULTILINE
        )    
        padrao = re.compile(
            r'(?<!\d)(\d+)\s*[\.\)]\s*'     # início (1), 2), etc.
            r'([\s\S]*?)'
            r'(?=\n?\s*(?:\d+\s*[\.\)])|\Z)',  # próxima reivindicação REAL
            re.DOTALL
        )
        padrao = re.compile(
            r'(?:^|\n)\s*(\d+)\s*[\.\)]\s*'
            r'([\s\S]*?)'
            r'(?=(?:\n\s*\d+\s*[\.\)])|\Z)',
            re.DOTALL
        )
        resultado = []
        for m in padrao.finditer(texto_reiv):
            numero = m.group(1)
            conteudo = m.group(2).strip()
            conteudo = (
                conteudo
                .replace('\r', ' ')
                .replace('\xa0', ' ')
                .replace('\ufeff', ' ')
                .replace('"',' ')
                .replace("'",' ')
                .replace('\n',' ')
            )
            resultado.append(f"{numero}. {conteudo}")
            
    if resultado is not None:
        return resultado 
    else:
        return None
        


print(extrair_reivindicacoes(texto))

['1. Metodo para reconhecimento do tipo de veiculo com base  em um escaner a laser, o metodo caracterizado pelo fa to de que compreende  as etapas de:   detectar que um veiculo a ser verificado entrou em uma area  de reconhecimento;   fazer com que um escaner a laser se mova em relacao ao  veiculo a ser verificado;']


In [312]:
texto = """
[0026]  O que foi revelado sao meramente algumas modalidades 
especificas da presente invencao, mas a presente inven cao nao esta restrita a 
estas, versados na tecnica podem fazer varias modificacoe s e variacoes na 
presente invencao sem fugir do espirito ou escopo da in vencao. Obviamente, 
todas modificacoes concebiveis aos versados na tecnic a devem se enquadrar 
no escopo de protecao da presente invencao. 1 / 5 
REIVINDICAC OES  
1. Metodo para reconhecimento do tipo de veiculo com base 
em um escaner a laser, o metodo caracterizado pelo fa to de que compreende 
as etapas de:  
detectar que um veiculo a ser verificado entrou em uma area 
de reconhecimento;  
fazer com que um escaner a laser se mova em relacao ao 
veiculo a ser verificado;  


"""

In [230]:
texto = """
112014014731
Petição depósito (200)
Data da Petição: 16/06/2014

REIVINDICACOES 
1) Sistema de comunicacoes para faturamento entre um 
tornar  outras o meio de pagamento destinado e originalmente para o transporte em meio de pagamento multiplo, ou seja,
finalidades alem do transporte, ou seja, para outras finalidades alem do 
transporte, com aplicacoes pre-pagas e pos-pagas com operacoes de
debito, credito, recarga, dentre outras, aplicando tecnologia
denominada dudl interface, que consiste num chip uUnico que possui as 
duas interfaces (uma com contato e a outra sem contato), aplicado a
profissional verificado; e 
2) método caracterizado pela 1 
RESUMO
"""

x = " "

In [232]:
texto = """
computador para realizar as etapas do 
metodo conforme definido em qualquer uma 
das reivindicacoes 9 a 16. Reivindicacao excluida.  

 REIVINDICACOES: 
1. METODO DE RASTREAMENTO DE COMUNICACOES ENTRE 
UM ASSINANTE E UM PROFISSIONAL, caracterizado por 
compreender: 
2. METODO DE RASTREAMENTO DE COMUNICACOES ENTRE 
UM ASSINANTE E UM PROFISSIONAL, caracterizado por 
compreender: 

METODO DE RASTREAMENTO DE COMUNICACOES ENTRE UM 
ASSINANTE E UM PROFISSIONAL 
FUNDAMENTOS 
[01] O negocio de muitos profissionais esta fornecendo respostas e 
conselheiros aos clientes. Cada vez mais, esses clientes podem estar 
em locais geograficamente
"""

In [54]:


texto = """
Conforme será apresentado no Quadro abaixo, onde são sinalizadas as alterações por 
meio de sublinhados (inclusão) e tachados (exclusões). Quanto às inclusões, na terceira coluna 
é citado o suporte no relatório descritivo depositado na Petição 870210071194, de 05/08/2021, 
estando todas as alterações rigorosamente compreendidas dentro do inicialmente proposto, 
conforme claramente demonstrado. 
Quanto às inclusões, na terceira coluna é citado o suporte no relatório descritivo 
depositado na Petição 870210071194, de 05/08/2021, estando todas as alterações rigorosamente 
compreendidas dentro do inicialmente proposto. 
REIVINDICAVÇÕES 
APRESENTADAS petição n. 
870200028162, de 02/03/2020 
NOVAS REIVINDICAÇÕES 
1.Sistema para faturamento de 
comunicações entre um assinante 
e um profissional verificado 
caracterizado pelo fato de que 
compreende: um servidor;  
um módulo de verificação em 
execução 
no 
servidor 
e 
configurado para verificar se o 
profissional 
verificado 
corresponde a um indivíduo 
procurado pelo assinante, ao:  - obter uma pluralidade de 
dados de verificação para o 
profissional 
verificado; 
verificar, 
1.Sistema para faturamento de 
comunicações entre um assinante 
(104) e um profissional verificado 
(108), 
caracterizado 
compreender: 
Um sistema de computador 
(1200); 
Um módulo de verificação (132) 
executado 
no sistema de 
computador (1200) e configurado 
para verificar se o profissional 
verificador (108) corresponde a 
um indivíduo procurado pelo 
assinante (104), ao: 
SUPORTE NO 
RELATÓRIO 
DESCRITIVO 
por 
com base na 
pluralidade de dados de 
verificação, pelo menos um 
selecionado do grupo que 
consiste em uma identidade 
Obter uma pluralidade de dados de 

[...]

[0131] Embora a invenção tenha sido descrita em relação a um número limitado 
de concretizações, os técnicos versados no assunto, tendo o benefício desta 
divulgação, apreciarão que outras concretizações podem ser criadas que não 
se afastem do escopo da invenção, conforme divulgado aqui em. Por 
conseguinte, o âmbito da invenção deve ser limitado apenas pelas 
reivindicações anexas. 
Petição 870230021203, de 13/03/2023, pág. 62/68

REIVINDICAÇÕES: 
1. 
Sistema de rastreamento e faturamento de comunicações entre um assinante 
(104) e um profissional verificado (108), caracterizado por compreender: 
a) Um sistema de computador (1200); 
b) Um módulo de verificação (132) executado no sistema de computador 
(1200) e configurado para verificar se o profissional verificador (108) 
corresponde a um indivíduo procurado pelo assinante (104), que: 
Obtém uma pluralidade de dados de verificação (202) para o profissional 
verificado (108); 
Verifica, com base na pluralidade de dados de verificação (202), pelo menos 
um selecionado do grupo que consiste em uma identidade do profissional 
verificado (108) e uma credencial do profissional verificado (108) para gerar 
um repositório profissional verificado (200) que corresponda ao indivíduo; 
Armazena, no repositório do profissional verificado (200), a pluralidade de 
dados de verificação (202) e uma identificação (ID) do dispositivo registrado 
associado a um dispositivo de comunicação utilizado pelo profissional 
verificado (108); e 


 """